#### Tools

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Bostonn?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Bostonn. First, I need to check if that's a real location. Wait, Bostonn doesn't seem right. The correct spelling should be Boston, Massachusetts. Maybe the user made a typo. Since the function requires a valid location, I should use the correct spelling. Let me call the get_weather function with Boston as the location. That should fetch the accurate weather information.\n", 'tool_calls': [{'id': 'pveh6g6x3', 'function': {'arguments': '{"location":"Boston, Massachusetts"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 155, 'total_tokens': 268, 'completion_time': 0.215713981, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.006361031, 'prompt_tokens_details': None, 'queue_time': 0.160132089, 'total_time': 0.222075012}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', '

##### Tool Execution loops

In [6]:
# step 1: Model generates test calls
messages = [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


# step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny right now. If you're there, it's a great day to enjoy outdoor activities! 🌞
